# Chapter 4 · A motion detector
The notebook version of [Chapter 4](../README.md): normalize, measure the wobble
in 1-second windows, choose a threshold from data, and check how often we are right.

---

# 第 4 章 · 動きの検出器
[第 4 章](../README.ja.md) のノートブック版です。正規化して、1 秒の窓でゆれを測り、
データからしきい値を決め、どれくらい当たるかを確かめます。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from csi_lab import simulate, simulate_sequence
from csi_lab.features import normalize_per_packet, motion_score, window_features, detect_motion, breathing_rate_bpm

## 1. Normalize
The receiver's automatic gain changes the volume for every packet. Dividing each packet by
its own average removes it.

---

## 1. 正規化
受信機の自動利得制御は、パケットごとに音量を変えます。パケットをそれ自身の平均で割ると、その影響が消えます。

In [ ]:
rec = simulate("still", duration_s=30, seed=1)
print("raw  :", rec.amplitude.mean(axis=1).std())
print("norm :", normalize_per_packet(rec.amplitude).mean(axis=1).std())

## 2. One motion number per second

---

## 2. 1 秒ごとに 1 つの動きの数

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for name in ["empty", "still", "walk", "wave"]:
    t, m = motion_score(simulate(name, duration_s=30, seed=1))
    ax.plot(t, m, marker=".", label=name)
ax.set_xlabel("time [s]"); ax.set_ylabel("motion"); ax.legend();

## 3. Threshold from a "nobody moves" recording

---

## 3. 「だれも動かない」記録からしきい値を決める

In [ ]:
_, still_motion = motion_score(simulate("still", duration_s=30, seed=1))
threshold = still_motion.max() * 1.2   # 20 % safety margin / 20 % の余裕
print("threshold =", round(threshold, 3))

## 4. Test on a story where the situation changes

---

## 4. 状況が移り変わる記録で試す

In [ ]:
story = simulate_sequence(
    [("empty", 20), ("walk", 20), ("still", 20), ("walk", 10), ("wave", 20), ("empty", 10)],
    seed=100,
)
w = window_features(story)
motion = w.features[:, w.names.index("motion")]
predicted = detect_motion(motion, threshold)
truth = np.isin(w.labels, ["walk", "wave"])
print("accuracy:", (predicted == truth).mean())
for label in ["empty", "still", "walk", "wave"]:
    sel = w.labels == label
    print(f"{label:6s} windows={sel.sum():3d}  said 'moving' in {predicted[sel].mean():.0%}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.step(w.start_s, motion, where="post", label="motion")
ax.axhline(threshold, color="red", ls="--", label="threshold")
ax.scatter(w.start_s[predicted != truth], motion[predicted != truth], color="red", zorder=3, label="wrong")
ax.set_xlabel("time [s]"); ax.legend();

## 5. Bonus: breathing (simulator only — not a medical device)

---

## 5. おまけ: 呼吸（シミュレーターだけ。医療機器ではありません）

In [ ]:
print("breathe:", round(breathing_rate_bpm(simulate("breathe", duration_s=60, seed=3)), 1), "per minute")
print("still  :", round(breathing_rate_bpm(simulate("still", duration_s=60, seed=2)), 1), "per minute  <- a number, but is it true?")

## 6. Your turn
- Lower the threshold (e.g. `* 1.0`) or raise it (`* 1.5`). Which mistakes go up, which go down?
- Try another `seed` for the story. Does the accuracy stay the same?

**Ask your AI:** "My detector misses hand waving. Ask me questions to help me find out why."

---

## 6. やってみよう
- しきい値を下げたり（例: `* 1.0`）上げたり（`* 1.5`）してみましょう。どのまちがいが増えて、どれが減りますか？
- お話の記録の `seed` を変えてみましょう。正解率は同じままですか？

**AI に聞いてみよう:**「わたしの検出器は手をふるのを見のがします。理由を見つけられるように質問してください。」